# 01 - Explore features

Sanity-checks the `features.py` pipeline on `data/sample.pgn`, then shows the shape of analysis you'd run on a real dataset (feature frequency vs. rating bin).

Run `jupyter lab` from the project root and open this notebook, or `code .` if you use VS Code's notebook support.

In [1]:
import sys
sys.path.append('../src')

import pandas as pd
from features import process_pgn_file

pd.set_option('display.max_columns', None)

## 1. Run the pipeline on the tiny sample file

`data/sample.pgn` has three synthetic games built to exercise castling, en passant, and promotion, so you can eyeball that every feature is computed correctly before pointing this at real data.

In [2]:
df = process_pgn_file('../data/sample.pgn')
df.shape

games: 4it [00:00, 875.32it/s]


(26, 28)

In [3]:
df.head(200)

,material_balance,white_has_bishop_pair,black_has_bishop_pair,white_kingside_rights,white_queenside_rights,black_kingside_rights,black_queenside_rights,is_castling_move,gives_check,is_en_passant_capture,en_passant_was_available,is_promotion,promotion_piece,is_capture,legal_moves_before,legal_moves_after,ply,move_number,side_to_move_was_white,player_elo,opponent_elo,white_elo,black_elo,result,time_control,termination,san,uci
0,0,True,True,True,True,True,True,False,False,False,False,False,NaN,False,20,20,1,1,True,2100,1950,2100,1950,*,600+0,Normal,e4,e2e4
1,0,True,True,True,True,True,True,False,False,False,False,False,NaN,False,20,29,2,1,False,1950,2100,2100,1950,*,600+0,Normal,e5,e7e5
2,0,True,True,True,True,True,True,False,False,False,False,False,NaN,False,29,29,3,2,True,2100,1950,2100,1950,*,600+0,Normal,Nf3,g1f3
3,0,True,True,True,True,True,True,False,False,False,False,False,NaN,False,29,27,4,2,False,1950,2100,2100,1950,*,600+0,Normal,Nc6,b8c6
4,0,True,True,True,True,True,True,False,False,False,False,False,NaN,False,27,31,5,3,True,2100,1950,2100,1950,*,600+0,Normal,Bc4,f1c4
5,0,True,True,True,True,True,True,False,False,False,False,False,NaN,False,31,33,6,3,False,1950,2100,2100,1950,*,600+0,Normal,Bc5,f8c5
6,0,True,True,False,False,True,True,True,False,False,False,False,NaN,False,33,36,7,4,True,2100,1950,2100,1950,*,600+0,Normal,O-O,e1g1
7,0,True,True,False,False,True,True,False,False,False,False,False,NaN,False,36,30,8,4,False,1950,2100,2100,1950,*,600+0,Normal,Nf6,g8f6
8,0,True,True,False,False,True,True,False,False,False,False,False,NaN,False,30,36,9,5,True,2100,1950,2100,1950,*,600+0,Normal,Re1,f1e1
9,0,True,True,False,False,False,False,True,False,False,False,False,NaN,False,36,32,10,5,False,1950,2100,2100,1950,*,600+0,Normal,O-O,e8g8


### Sanity checks

- Game 1 should show `is_castling_move=True` twice (moves 4 and 5), and `gives_check=True` on ply 21 (`Nxe7+`).
- Game 2 should show `is_en_passant_capture=True` on ply 5 (`exd6`).
- Game 3 should show `is_promotion=True`, `promotion_piece='Q'` on ply 1.

In [4]:
print('castling moves:', df['is_castling_move'].sum())
print('checks:', df['gives_check'].sum())
print('en passant captures:', df['is_en_passant_capture'].sum())
print('promotions:', df['is_promotion'].sum())
df[df['is_en_passant_capture']]

castling moves: 2
checks: 2
en passant captures: 1
promotions: 1


,material_balance,white_has_bishop_pair,black_has_bishop_pair,white_kingside_rights,white_queenside_rights,black_kingside_rights,black_queenside_rights,is_castling_move,gives_check,is_en_passant_capture,en_passant_was_available,is_promotion,promotion_piece,is_capture,legal_moves_before,legal_moves_after,ply,move_number,side_to_move_was_white,player_elo,opponent_elo,white_elo,black_elo,result,time_control,termination,san,uci
21,1,True,True,True,True,True,True,False,False,True,True,False,NaN,True,31,28,5,3,True,1400,1350,1400,1350,*,600+0,Normal,exd6,e5d6


## 2. Once this looks right: point it at real data

Download a monthly dump from https://database.lichess.org/ (`.pgn.zst`). Don't decompress the whole thing to disk -
use `process_pgn_zst_stream` from `features.py`, which streams through it and stops after `max_games`.

```python
from features import process_pgn_zst_stream

df = process_pgn_zst_stream(
    '../data/lichess_db_standard_rated_2026-01.pgn.zst',
    max_games=20000,
    min_elo=None,
)
df.to_parquet('../data/features_sample.parquet')
```

In [5]:

from features import process_pgn_zst_stream

df = process_pgn_zst_stream(
    '../data/lichess_db_standard_rated_2026-08.pgn.zst',
    max_games=20000,
    min_elo=None,
)
df.to_parquet('../data/features_sample.parquet')

games: 100%|██████████| 20000/20000 [02:20<00:00, 142.48it/s]


## 3. First correlation pass (template)

Shape of the analysis you'll want once you have real data: does a feature's frequency change across rating bins?

In [7]:
# df = pd.read_parquet('../data/features_sample.parquet')  # real data, once you have it

bins = [0, 1200, 1600, 2000, 4000]
labels = ['<1200', '1200-1600', '1600-2000', '2000+']
df['elo_bin'] = pd.cut(df['player_elo'], bins=bins, labels=labels)

# example: how often does each rating band play a move that gives check?
df.groupby('elo_bin', observed=True)['gives_check'].mean()

elo_bin
<1200        0.088609
1200-1600    0.075827
1600-2000    0.070111
2000+        0.069557
Name: gives_check, dtype: float64

In [6]:
from scipy.stats import pointbiserialr

# example: correlation between player rating and giving check on a given move
valid = df.dropna(subset=['player_elo'])
corr, pval = pointbiserialr(valid['gives_check'], valid['player_elo'])
print(f'point-biserial r={corr:.3f}, p={pval:.4f}')

point-biserial r=-0.019, p=0.9278
